# Week 2, Sat: pose pilot on 60 vault clips

**Before running:** put `posekit.py` in `My Drive/capstone/code/`. **Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*.

What it does, for each of two pose models (YOLO26-L and YOLO11-L pose):
1. finds every person and their 17 body keypoints in every frame of 60 clips (60 different competitions)
2. picks the gymnast (offline, from the saved detections) and measures pose quality per stage
3. draws a **contact sheet** per clip (approach / flight / landing, 5 frames each, cropped on the gymnast) and **overlay videos** for 10 clips
4. writes a review checklist for you to fill in by eye

Everything is saved under `My Drive/capstone/pose/`. Resumable: re-run after a disconnect and finished clips are skipped.
Expected time on a T4: ~5 min copying, ~10-20 min per model, ~5 min drawing.

`MODE = 'full'` (Monday) runs the chosen model on all clips instead.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics

/bin/bash: line 1: nvidia-smi: command not found


## 1. Settings and paths

In [ ]:
MODE    = 'full'                                  # 'pilot' (60 clips) or 'full' (all clips)
MODELS  = ['yolo26l-pose.pt']   # first = primary (contact sheets you review)
IMGSZ   = 960      # inference size; clips are <=480p, so 960 upsamples small, far-away gymnasts
CONF    = 0.10     # low on purpose: flipping bodies get low person scores; the gymnast picker filters
N_PILOT = 60
N_VIDEOS_OVERLAY = 10
VERSION = 'v3'     # v2 = per-stage picking (camera cuts); v3 = v2 + rotated pass (upside-down frames)
ROTATE  = True     # second detection pass on rotated frames where the gymnast is missing

from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, time, json
from pathlib import Path
ROOT      = Path('/content/drive/MyDrive/capstone')
CLIPS_DRV = ROOT / 'clips'
CSV       = ROOT / 'out' / 'clips_available.csv'
POSE      = ROOT / 'pose'
LOCAL     = Path('/content/clips')
sys.path.insert(0, str(ROOT / 'code'))
assert (ROOT / 'code' / 'posekit.py').exists(), 'put posekit.py in My Drive/capstone/code/ first'
import posekit as pk
tag = lambda m: Path(m).stem.replace('-pose', '') + f'_{IMGSZ}'
print('ok', [tag(m) for m in MODELS])

Mounted at /content/drive
ok ['yolo26l_960']


## 2. Clip list: pilot = 60 clips with stage annotations, one per competition where possible

In [ ]:
import numpy as np, pandas as pd
df = pd.read_csv(CSV, dtype={'vault_code': str, 'bib': str})
df = df[df.split.isin(['train', 'test'])].reset_index(drop=True)
df['clip_id'] = df.video_id + '__' + df.event_key
if MODE == 'pilot':
    cand = df[df.segment_key.notna()].sample(frac=1, random_state=42)
    picks, used = [], {}
    while len(picks) < N_PILOT and len(picks) < len(cand):          # round-robin over competitions
        for v, g in cand.groupby('video_id', sort=False):
            g = g[~g.clip_id.isin(picks)]
            if len(g) and len(picks) < N_PILOT:
                picks.append(g.clip_id.iloc[0])
    clips = df[df.clip_id.isin(picks)].reset_index(drop=True)
else:
    clips = df
POSE.mkdir(parents=True, exist_ok=True)
clips.to_csv(POSE / f'{MODE}_clips.csv', index=False)
print(len(clips), 'clips from', clips.video_id.nunique(), 'competitions |', clips.split.value_counts().to_dict())

LOCAL.mkdir(exist_ok=True)
for f in clips.clip_file:
    if not (LOCAL / f).exists():
        shutil.copy(CLIPS_DRV / f, LOCAL / f)
print('local clips:', len(list(LOCAL.glob('*.mp4'))))

961 clips from 97 competitions | {'train': 784, 'test': 177}
local clips: 961


## 3. Detect people and keypoints (all people, all frames; saved raw)
The gymnast is chosen later from these raw detections, so the choice can be re-tuned without the GPU.

In [ ]:
import cv2, torch
from ultralytics import YOLO
from tqdm.auto import tqdm

def detect(model, path):
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0; cap.release()
    per_frame, W, H = [], None, None
    for r in model.predict(str(path), stream=True, conf=CONF, imgsz=IMGSZ, max_det=10, verbose=False):
        H, W = r.orig_shape
        per_frame.append((r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(),
                          r.keypoints.data.cpu().numpy() if r.keypoints is not None else np.zeros((0, 17, 3))))
    return pk.pack_raw(per_frame, fps, W, H)

for m in MODELS:
    out = POSE / tag(m) / 'raw'; out.mkdir(parents=True, exist_ok=True)
    todo = [r for r in clips.itertuples() if not (out / f'{r.clip_id}.npz').exists()]
    print(f'{tag(m)}: {len(clips) - len(todo)} done, {len(todo)} to go')
    if not todo:
        continue
    model = YOLO(m)
    t0, nf, errs = time.time(), 0, []
    for r in tqdm(todo):
        try:
            raw = detect(model, LOCAL / r.clip_file)
            np.savez_compressed(out / f'{r.clip_id}.npz', **raw); nf += int(raw['n_frames'])
        except Exception as e:
            errs.append((r.clip_id, repr(e)[:150]))
    dt = time.time() - t0
    print(f'{tag(m)}: {nf} frames in {dt/60:.1f} min ({nf/max(dt,1):.0f} fps) | errors {len(errs)}', errs[:5])
    del model; torch.cuda.empty_cache()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
yolo26l_960: 961 done, 0 to go


## 3b. Second pass for upside-down frames
Pose models are trained on upright people and miss the handstand on the table and the inverted second flight.
For frames in the flight and landing windows where v2 found no gymnast, the frame is rotated 180°, 90° and 270°, detected again, and the results rotated back. Saved to `raw_rot/`; only these frames are processed, so it is quick.

In [ ]:
def read_frames(path, wanted):
    cap, out, i = cv2.VideoCapture(str(path)), {}, 0
    wanted = set(int(w) for w in wanted)
    while wanted and i <= max(wanted):
        ok, f = cap.read()
        if not ok: break
        if i in wanted: out[i] = f
        i += 1
    cap.release(); return out

CV_ROT = {180: cv2.ROTATE_180, 90: cv2.ROTATE_90_CLOCKWISE, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}

def rotated_pass(model, r, raw):
    fr = pk.stage_frames(r._asdict(), float(raw['fps']), int(raw['n_frames']))
    idx, _ = pk.select_gymnast(raw, fr)
    need = sorted({f for st in ('flight', 'landing') if st in fr for f in range(*fr[st]) if idx[f] < 0})
    W, H = int(raw['width']), int(raw['height'])
    frames = read_frames(LOCAL / r.clip_file, need)
    fs = [f for f in need if f in frames]
    per = {f: [] for f in fs}
    for rot, cvr in CV_ROT.items():
        imgs = [cv2.rotate(frames[f], cvr) for f in fs]
        for i in range(0, len(imgs), 16):
            for f, rr in zip(fs[i:i + 16], model.predict(imgs[i:i + 16], conf=CONF, imgsz=IMGSZ, max_det=5, verbose=False)):
                b, k = pk.unrotate_dets(rr.boxes.xyxy.cpu().numpy(), rr.keypoints.data.cpu().numpy(), rot, W, H)
                per[f].append((b, rr.boxes.conf.cpu().numpy(), k))
    F, B, C, K = [], [], [], []
    for f, lst in per.items():
        b = np.concatenate([x[0] for x in lst]) if lst else np.zeros((0, 4))
        if not len(b): continue
        c = np.concatenate([x[1] for x in lst]); k = np.concatenate([x[2] for x in lst])
        keep = list(pk.nms(b, c))                                     # merge the three rotations
        orig = raw['det_box'][raw['det_frame'] == f]                  # drop people already found upright
        if len(orig):
            keep = [i for i in keep if pk._iou(b[i:i + 1], orig).max() < 0.5]
        if keep:
            F.append(np.full(len(keep), f)); B.append(b[keep]); C.append(c[keep]); K.append(k[keep])
    cat = lambda xs, shape: np.concatenate(xs) if xs else np.zeros(shape)
    return dict(det_frame=cat(F, (0,)).astype(np.int32), det_box=cat(B, (0, 4)).astype(np.float32),
                det_conf=cat(C, (0,)).astype(np.float32), det_kpt=cat(K, (0, 17, 3)).astype(np.float16),
                n_need=np.int32(len(need)))

def load_merged(base, clip_id):
    raw = pk.load_raw(base / 'raw' / f'{clip_id}.npz')
    rf = base / 'raw_rot' / f'{clip_id}.npz'
    return pk.merge_raw(raw, pk.load_raw(rf) if (ROTATE and rf.exists()) else None)

if False:
    for m in MODELS:
        base = POSE / tag(m); out = base / 'raw_rot'; out.mkdir(exist_ok=True)
        todo = [r for r in clips.itertuples() if (base / 'raw' / f'{r.clip_id}.npz').exists()
                and not (out / f'{r.clip_id}.npz').exists()]
        print(f'{tag(m)} rotated pass: {len(todo)} clips to go')
        if not todo: continue
        model = YOLO(m); t0, n_need, n_new = time.time(), 0, 0
        for r in tqdm(todo):
            ex = rotated_pass(model, r, pk.load_raw(base / 'raw' / f'{r.clip_id}.npz'))
            np.savez_compressed(out / f'{r.clip_id}.npz', **ex)
            n_need += int(ex['n_need']); n_new += len(np.unique(ex['det_frame']))
        print(f'  {n_need} frames re-checked, new detections in {n_new} ({n_new / max(1, n_need):.0%}) '
              f'| {(time.time() - t0) / 60:.1f} min')
        del model; torch.cuda.empty_cache()

## 4. Pick the gymnast, measure quality per stage, first symbols
Quality columns per stage: `coverage` (frames with the gymnast found), `lower_conf` (hip/knee/ankle confidence), `lower_visible`, `jitter` (keypoint jump per frame, in body heights), `limb_cv` (thigh/shin length variation; large = keypoints swapping or garbage), `box_h` (gymnast height / frame height).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
SUMMARY = {}
for m in MODELS:
    base = POSE / tag(m); (base / f'gymnast_{VERSION}').mkdir(exist_ok=True)
    rows = []
    for r in tqdm(clips.itertuples(), total=len(clips), desc=tag(m)):
        f = base / 'raw' / f'{r.clip_id}.npz'
        if not f.exists():
            continue
        raw = load_merged(base, r.clip_id)
        fr = pk.stage_frames(r._asdict(), float(raw['fps']), int(raw['n_frames']))
        idx, info = pk.select_gymnast(raw, fr)
        k, b = pk.gymnast_arrays(raw, idx)
        np.savez_compressed(base / f'gymnast_{VERSION}' / f'{r.clip_id}.npz', kpts=k, boxes=b, det_idx=idx,
                            fps=raw['fps'], stage_frames=json.dumps(fr))
        row = dict(clip_id=r.clip_id, video_id=r.video_id, vault_code=r.vault_code, E=r.E, split=r.split,
                   n_frames=int(raw['n_frames']), n_tracks=info['n_tracks'], per_stage=json.dumps(info.get('per_stage', {})),
                   top_tracks=json.dumps(info.get('top', [])))
        for st in ['whole'] + pk.STAGES:
            if st in fr:
                for q, v in pk.quality(k, b, *fr[st], float(raw['height'])).items():
                    row[f'{st}_{q}'] = v
                # was ANY reasonably sized person detected? (separates detector misses from picker misses)
                h = raw['det_box'][:, 3] - raw['det_box'][:, 1]
                big = raw['det_frame'][h >= 0.08 * float(raw['height'])]
                lo, hi = fr[st]
                row[f'{st}_any_det'] = len(np.unique(big[(big >= lo) & (big < hi)])) / max(1, hi - lo)
                sel = idx[lo:hi][idx[lo:hi] >= 0]
                row[f'{st}_from_rotated'] = float(raw['det_rot'][sel].mean()) if len(sel) else 0.0
        row.update(pk.symbols(k, b, fr))
        rows.append(row)
    S = pd.DataFrame(rows); S.to_csv(base / f'summary_{VERSION}.csv', index=False); SUMMARY[m] = S

def overview(S):
    t = {}
    for st in ['approach', 'flight', 'landing']:
        t[st] = {'coverage (median)': S[f'{st}_coverage'].median(),
                 'clips coverage>=0.8': (S[f'{st}_coverage'] >= 0.8).mean(),
                 'share from rotated pass': S[f'{st}_from_rotated'].mean(),
                 'lower-body conf': S[f'{st}_lower_conf'].mean(),
                 'limb_cv (median)': S[f'{st}_limb_cv'].median(),
                 'jitter (median)': S[f'{st}_jitter'].median()}
    return pd.DataFrame(t).round(3)

for m, S in SUMMARY.items():
    print(f'\n=== {tag(m)} ({len(S)} clips) | gymnast height / frame: {S.whole_box_h.median():.2f}')
    print(overview(S).to_string())

yolo26l_960:   0%|          | 0/961 [00:00<?, ?it/s]


=== yolo26l_960 (961 clips) | gymnast height / frame: 0.61
                         approach  flight  landing
coverage (median)           0.982   0.756    0.791
clips coverage>=0.8         0.717   0.376    0.443
share from rotated pass     0.000   0.112    0.101
lower-body conf             0.676   0.735    0.840
limb_cv (median)            0.194   0.237    0.177
jitter (median)             0.028   0.067    0.022


## 5. Contact sheets (one PNG per clip, both models) and overlay videos (primary model)
Each sheet: 3 rows (approach / flight / landing) × 5 frames, cropped on the chosen person, skeleton drawn.
A red "NOT FOUND" tile means no gymnast in that frame.

In [ ]:
import subprocess

def tile(img, kp, box, size=200):
    img = img.copy()
    if box is None or not np.isfinite(box).all():
        t = cv2.resize(img, (size, size)); cv2.rectangle(t, (0, 0), (size - 1, size - 1), (0, 0, 255), 4)
        cv2.putText(t, 'NOT FOUND', (10, size // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2); return t
    pk.draw_skeleton(img, kp)
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    s = min(max(box[2] - box[0], box[3] - box[1]) * 1.6 / 2 + 20, max(img.shape[:2]) / 2)
    pad = int(s) + 1                                   # pad so the crop stays square at frame edges
    img = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(40, 40, 40))
    x0, y0 = int(cx - s) + pad, int(cy - s) + pad
    return cv2.resize(img[y0:y0 + int(2 * s), x0:x0 + int(2 * s)], (size, size))

def sheet(r, base, n=5):
    g = np.load(base / f'gymnast_{VERSION}' / f'{r.clip_id}.npz'); k, b = g['kpts'], g['boxes']
    fr = json.loads(str(g['stage_frames']))
    picks = {st: np.linspace(fr[st][0], fr[st][1] - 1, n).round().astype(int) for st in pk.STAGES if st in fr}
    frames = read_frames(LOCAL / r.clip_file, np.concatenate(list(picks.values())))
    rows = []
    for st, fs in picks.items():
        ts = [tile(frames[f], k[f], b[f]) if f in frames else np.zeros((200, 200, 3), np.uint8) for f in fs]
        lab = np.full((200, 110, 3), 255, np.uint8)
        cv2.putText(lab, st, (5, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        rows.append(np.hstack([lab] + ts))
    img = np.vstack(rows)
    head = np.full((40, img.shape[1], 3), 255, np.uint8)
    cv2.putText(head, f'{r.clip_id} | vault {r.vault_code} | E {r.E}', (5, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
    return np.vstack([head, img])

def overlay(r, raw, base, out_path):
    g = np.load(base / f'gymnast_{VERSION}' / f'{r.clip_id}.npz'); k, b = g['kpts'], g['boxes']
    fr = json.loads(str(g['stage_frames']))
    cap = cv2.VideoCapture(str(LOCAL / r.clip_file)); fps = cap.get(cv2.CAP_PROP_FPS) or 25
    tmp = str(out_path) + '.tmp.mp4'; vw = None; i = 0
    while True:
        ok, f = cap.read()
        if not ok: break
        if vw is None: vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), fps, (f.shape[1], f.shape[0]))
        for bb in raw['det_box'][raw['det_frame'] == i]:
            cv2.rectangle(f, tuple(int(v) for v in bb[:2]), tuple(int(v) for v in bb[2:]), (160, 160, 160), 1)
        if i < len(b) and np.isfinite(b[i]).all():
            cv2.rectangle(f, tuple(int(v) for v in b[i, :2]), tuple(int(v) for v in b[i, 2:]), (0, 255, 0), 2)
            pk.draw_skeleton(f, k[i])
        st = next((s for s in pk.STAGES if s in fr and fr[s][0] <= i < fr[s][1]), '')
        cv2.putText(f, f'{i} {st}', (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        vw.write(f); i += 1
    cap.release(); vw.release()
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', tmp, '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(out_path)])
    Path(tmp).unlink(missing_ok=True)

if MODE == 'pilot':
    for m in MODELS:
        base = POSE / tag(m); (base / f'sheets_{VERSION}').mkdir(exist_ok=True)
        for r in tqdm(clips.itertuples(), total=len(clips), desc=f'sheets {tag(m)}'):
            p = base / f'sheets_{VERSION}' / f'{r.clip_id}.png'
            if not p.exists() and (base / f'gymnast_{VERSION}' / f'{r.clip_id}.npz').exists():
                cv2.imwrite(str(p), sheet(r, base))
    base = POSE / tag(MODELS[0]); (base / f'overlays_{VERSION}').mkdir(exist_ok=True)
    for r in tqdm(list(clips.itertuples())[:N_VIDEOS_OVERLAY], desc='overlays'):
        p = base / f'overlays_{VERSION}' / f'{r.clip_id}.mp4'
        if not p.exists():
            overlay(r, load_merged(base, r.clip_id), base, p)
    print('sheets and overlays written under', POSE)

## 6. Review checklist (fill in by eye, Sunday)
Written once to `pose/review_checklist_<version>.csv`; re-running never overwrites your answers.

In [ ]:
from IPython.display import Image, display
rev = POSE / f'review_checklist_{VERSION}.csv'
if MODE == 'pilot' and not rev.exists():
    c = clips[['clip_id', 'video_id', 'vault_code', 'E']].copy()
    c['sheet'] = [f'{tag(MODELS[0])}/sheets_{VERSION}/{x}.png' for x in c.clip_id]
    for col in ['gymnast_right', 'approach_ok', 'flight_ok', 'landing_ok', 'notes']:
        c[col] = ''
    c.to_csv(rev, index=False); print('written', rev)
else:
    print('checklist exists, not overwritten:', rev)
display(Image(str(POSE / tag(MODELS[0]) / f'sheets_{VERSION}' / f'{clips.clip_id.iloc[0]}.png')))

checklist exists, not overwritten: /content/drive/MyDrive/capstone/pose/review_checklist_v3.csv


FileNotFoundError: No such file or directory: '/content/drive/MyDrive/capstone/pose/yolo26l_960/sheets_v3/1JsRXIoR3C0__E_000598_000605.png'

FileNotFoundError: No such file or directory: '/content/drive/MyDrive/capstone/pose/yolo26l_960/sheets_v3/1JsRXIoR3C0__E_000598_000605.png'

<IPython.core.display.Image object>